# AuditFlow — Regression Test Suite

Automated verification of core AuditFlow functionality.
Covers TC-0 through TC-4 from [DEVELOPERS.md](../DEVELOPERS.md).

**Before running:**
```bash
just verify
# Wait ~60 s for all containers to reach "healthy" status
```

**Requirements:** `pip install jupyter requests`

Run all cells top-to-bottom with **Kernel → Restart & Run All**.



## Section 0 — Setup & Preflight

In [ ]:
import requests
import subprocess
import json
import time
from datetime import datetime, timezone

# ── Service endpoints ─────────────────────────────────────────
BACKEND_API      = "http://localhost:8080/api/v1"
BACKEND_ACTUATOR = "http://localhost:8080/actuator"
TRANSFORMER      = "http://localhost:8081"
SINK             = "http://localhost:8082"
RABBITMQ_API     = "http://localhost:15673/api"

# ── Container names for docker logs ───────────────────────────
SINK_CONTAINER    = "auditflow-sink"
BACKEND_CONTAINER = "auditflow-be"

# ── Timing — increase ASYNC_WAIT_S on slow machines ───────────
ASYNC_WAIT_S = 4
HTTP_TIMEOUT = 10

# ── Result collector ──────────────────────────────────────────
_results = {}   # tc_key → (passed: bool, detail: str)

print("✓ Configuration loaded")

In [ ]:
def docker_logs_since(container: str, since: datetime) -> list:
    """Fetch container logs (stdout + stderr) since the given UTC datetime."""
    since_str = since.strftime("%Y-%m-%dT%H:%M:%SZ")
    result = subprocess.run(
        ["docker", "logs", container, "--since", since_str],
        capture_output=True, text=True
    )
    return (result.stdout + result.stderr).splitlines()


def assert_pass(tc_key: str, condition: bool, detail: str):
    """Record result; raise AssertionError on failure so the cell stops."""
    _results[tc_key] = (condition, detail)
    if condition:
        print(f"  ✓  {tc_key}: {detail}")
    else:
        raise AssertionError(f"  ✗  {tc_key}: {detail}")


def display_response(resp):
    print(f"  Status: {resp.status_code}")
    try:
        print(json.dumps(resp.json(), indent=2))
    except Exception:
        print(f"  Body: {resp.text[:500]}")


print("✓ Helpers defined")

In [ ]:
print("── Preflight: health checks ─────────────────────────────────")
for name, url in [
    ("backend",     f"{BACKEND_ACTUATOR}/health"),
    ("transformer", f"{TRANSFORMER}/health"),
    ("sink",        f"{SINK}/health"),
]:
    try:
        resp = requests.get(url, timeout=HTTP_TIMEOUT)
        ok = resp.status_code == 200
        body = resp.json() if ok else {}
        up = body.get("status") == "UP"
        detail = f"{name} is UP" if (ok and up) else f"HTTP {resp.status_code}, status={body.get('status')}"
        assert_pass(f"TC-0.3 {name}", ok and up, detail)
    except requests.exceptions.ConnectionError:
        assert_pass(f"TC-0.3 {name}", False,
            f"{name} unreachable at {url} — run `just verify` first")


In [ ]:
print("── Preflight: plugin registries ────────────────────────────")
t_resp = requests.get(f"{TRANSFORMER}/registry", timeout=HTTP_TIMEOUT)
assert t_resp.status_code == 200, f"Transformer registry returned {t_resp.status_code}"
transformer_ids = [t["id"] for t in t_resp.json().get("transformers", [])]
assert_pass("TC-0.4 transformer registry",
    "zero" in transformer_ids,
    f"'zero' in {transformer_ids}")

s_resp = requests.get(f"{SINK}/registry", timeout=HTTP_TIMEOUT)
assert s_resp.status_code == 200, f"Sink registry returned {s_resp.status_code}"
sink_ids = [s["id"] for s in s_resp.json().get("sinks", [])]
assert_pass("TC-0.4 sink registry",
    "logging_sink" in sink_ids,
    f"'logging_sink' in {sink_ids}")

## TC-1 — Happy-Path Event Flow

Verifies an event published via REST flows through RabbitMQ, the `zero` transformer,
and is delivered by `logging_sink`.

In [ ]:
print("── TC-1: Happy-Path Event Flow ─────────────────────────────")
_tc1_ts = datetime.now(timezone.utc)

TC1_EVENT = {
    "eventId":      "00000000-0000-0000-0001-000000000001",
    "eventType":    "user.login",
    "sourceSystem": "auth-service",
    "tenantId":     "T-VERIFY-001",
    "extra": {
        "userId":        "alice",
        "sessionId":     "sess-abc-123",
        "action_status": "SUCCESS"
    }
}

resp = requests.post(f"{BACKEND_API}/audit/publish", json=TC1_EVENT, timeout=HTTP_TIMEOUT)
display_response(resp)
assert_pass("TC-1.1 publish", resp.status_code == 200,
    f"POST /audit/publish → {resp.status_code}")

In [ ]:
print(f"  Waiting {ASYNC_WAIT_S}s for async pipeline …")
time.sleep(ASYNC_WAIT_S)

logs = docker_logs_since(SINK_CONTAINER, _tc1_ts)
matching = [l for l in logs if "user.login" in l]
print(f"  Sink lines containing 'user.login': {len(matching)}")
for l in matching[:3]:
    print(f"    {l}")

assert_pass("TC-1.2 sink delivery",
    len(matching) >= 1,
    f"{len(matching)} sink log line(s) with 'user.login'")

## TC-2 — Pipeline Condition Filtering

Verifies that pipeline 1 (`security-alerts`, condition `eventType = security.alert`) fires only
for matching events. Pipeline 0 (`e2e-logging`) fires for every event.

In [ ]:
print("── TC-2a: Non-matching event (api.call) ────────────────────")
_tc2a_ts = datetime.now(timezone.utc)

TC2A_EVENT = {
    "eventId":      "00000000-0000-0000-0002-000000000001",
    "eventType":    "api.call",
    "sourceSystem": "backend-service"
}

resp = requests.post(f"{BACKEND_API}/audit/publish", json=TC2A_EVENT, timeout=HTTP_TIMEOUT)
assert resp.status_code == 200
time.sleep(ASYNC_WAIT_S)

logs = docker_logs_since(SINK_CONTAINER, _tc2a_ts)
matching = [l for l in logs if "api.call" in l]
print(f"  Lines with 'api.call': {len(matching)}")
for l in matching[:3]:
    print(f"    {l}")

assert_pass("TC-2.1 condition miss (fired)",
    len(matching) >= 1,
    f"pipeline 0 fired: {len(matching)} line(s)")

has_warning = any("WARNING" in l or "WARN" in l for l in matching)
assert_pass("TC-2.1 condition miss (no WARN)",
    not has_warning,
    "pipeline 1 did NOT fire (no WARNING in sink log)")

In [ ]:
print("── TC-2b: Matching event (security.alert) ──────────────────")
_tc2b_ts = datetime.now(timezone.utc)

TC2B_EVENT = {
    "eventId":      "00000000-0000-0000-0002-000000000002",
    "eventType":    "security.alert",
    "sourceSystem": "auth-service",
    "extra":        {"action_status": "FAILURE"}
}

resp = requests.post(f"{BACKEND_API}/audit/publish", json=TC2B_EVENT, timeout=HTTP_TIMEOUT)
assert resp.status_code == 200
time.sleep(ASYNC_WAIT_S)

logs = docker_logs_since(SINK_CONTAINER, _tc2b_ts)
matching = [l for l in logs if "security.alert" in l]
print(f"  Lines with 'security.alert': {len(matching)}")
for l in matching[:5]:
    print(f"    {l}")

assert_pass("TC-2.2 condition match (count)",
    len(matching) >= 2,
    f"{len(matching)} sink entries — both pipelines fired")

has_warning = any("WARNING" in l or "WARN" in l for l in matching)
assert_pass("TC-2.2 condition match (WARN)",
    has_warning,
    "pipeline 1 logged at WARNING level")

## TC-3 — PII / Sensitive Data Redaction

Verifies that `extra.userId` is masked to `***` and `extra.sessionId` is dropped before
any downstream sink receives the event.

In [ ]:
print("── TC-3: PII / Sensitive Data Redaction ────────────────────")
_tc3_ts = datetime.now(timezone.utc)

TC3_EVENT = {
    "eventId":      "00000000-0000-0000-0003-000000000001",
    "eventType":    "user.profile.updated",
    "sourceSystem": "profile-service",
    "extra": {
        "userId":    "alice",
        "sessionId": "sess-secret-999",
        "action":    "update_email"
    }
}

resp = requests.post(f"{BACKEND_API}/audit/publish", json=TC3_EVENT, timeout=HTTP_TIMEOUT)
display_response(resp)
assert_pass("TC-3.1 redaction publish", resp.status_code == 200,
    f"POST /audit/publish → {resp.status_code}")

In [ ]:
time.sleep(ASYNC_WAIT_S)

logs = docker_logs_since(SINK_CONTAINER, _tc3_ts)
matching = [l for l in logs if "user.profile.updated" in l]
print(f"  Lines with 'user.profile.updated': {len(matching)}")
for l in matching[:3]:
    print(f"    {l}")

combined = " ".join(matching)

assert_pass("TC-3.2 userId masked",     "***" in combined,              "'***' found — userId masked")
assert_pass("TC-3.2 userId raw absent", "alice" not in combined,         "raw 'alice' absent from sink")
assert_pass("TC-3.2 sessionId dropped", "sess-secret-999" not in combined, "'sess-secret-999' absent")
assert_pass("TC-3.2 action preserved",  "update_email" in combined,     "'update_email' present")

## TC-4 — Idempotency (Duplicate Suppression)

Verifies that publishing the same `eventId` twice results in exactly one sink delivery.
The API returns 200 for both requests; deduplication happens asynchronously at the consumer.

In [ ]:
print("── TC-4: Idempotency ────────────────────────────────────────")
_tc4_ts = datetime.now(timezone.utc)

TC4_EVENT = {
    "eventId":      "00000000-0000-0000-0004-000000000001",
    "eventType":    "order.created",
    "sourceSystem": "order-service",
    "tenantId":     "T-VERIFY-004"
}

resp = requests.post(f"{BACKEND_API}/audit/publish", json=TC4_EVENT, timeout=HTTP_TIMEOUT)
display_response(resp)
assert_pass("TC-4.1 first publish", resp.status_code == 200,
    f"first publish → {resp.status_code}")

time.sleep(ASYNC_WAIT_S)

In [ ]:
print("  Publishing duplicate …")
resp2 = requests.post(f"{BACKEND_API}/audit/publish", json=TC4_EVENT, timeout=HTTP_TIMEOUT)
display_response(resp2)
assert_pass("TC-4.2 duplicate publish", resp2.status_code == 200,
    f"duplicate → {resp2.status_code} (API always accepts)")

time.sleep(ASYNC_WAIT_S)

In [ ]:
be_logs = docker_logs_since(BACKEND_CONTAINER, _tc4_ts)
eventId = TC4_EVENT["eventId"]
dedup_lines = [
    l for l in be_logs
    if eventId in l
    and any(m in l for m in ("duplicate", "Duplicate", "already", "idempoten", "DEDUPLICATED"))
]
print(f"  Backend dedup log lines: {len(dedup_lines)}")
for l in dedup_lines[:3]:
    print(f"    {l}")

assert_pass("TC-4.3 dedup log",
    len(dedup_lines) >= 1,
    f"{len(dedup_lines)} dedup rejection line(s) in backend log")

In [ ]:
sink_logs = docker_logs_since(SINK_CONTAINER, _tc4_ts)
delivered = [l for l in sink_logs if "order.created" in l and "T-VERIFY-004" in l]
print(f"  Sink deliveries for order.created / T-VERIFY-004: {len(delivered)}")
for l in delivered:
    print(f"    {l}")

assert_pass("TC-4.4 exactly one delivery",
    len(delivered) == 1,
    f"exactly 1 delivery (got {len(delivered)})")

## Extras (Optional)

Informational checks — do not contribute to the summary pass/fail count.

In [ ]:
# OPTIONAL — DLQ inspection
print("── DLQ inspection ───────────────────────────────────────────")
resp = requests.get(f"{BACKEND_ACTUATOR}/dlq", timeout=HTTP_TIMEOUT)
display_response(resp)

In [ ]:
# OPTIONAL — Consumer metrics spot-check
print("── Consumer metrics ─────────────────────────────────────────")
resp = requests.get(
    f"{BACKEND_ACTUATOR}/metrics/auditflow.consumer.events.processed",
    timeout=HTTP_TIMEOUT
)
display_response(resp)
try:
    value = resp.json()["measurements"][0]["value"]
    print(f"  events.processed = {value}")
    assert value > 0
    print("  ✓ metric is non-zero")
except Exception as e:
    print(f"  ⚠ could not verify metric: {e}")

In [ ]:
# OPTIONAL — RabbitMQ queue depth
print("── RabbitMQ queue depth ─────────────────────────────────────")
queue_name = "labs64-audit-topic.labs64.io-auditflow"
url = f"{RABBITMQ_API}/queues/%2F/{queue_name}"
resp = requests.get(url, auth=("guest", "guest"), timeout=HTTP_TIMEOUT)
display_response(resp)
try:
    ready = resp.json()["messages_ready"]
    print(f"  messages_ready = {ready}")
    assert ready == 0
    print("  ✓ queue is empty — all events consumed")
except Exception as e:
    print(f"  ⚠ queue check: {e}")

## Summary

In [ ]:
TC_LABELS = [
    ("TC-0.3 backend",                "TC-0.3  Backend health"),
    ("TC-0.3 transformer",            "TC-0.3  Transformer health"),
    ("TC-0.3 sink",                   "TC-0.3  Sink health"),
    ("TC-0.4 transformer registry",   "TC-0.4  Transformer registry"),
    ("TC-0.4 sink registry",          "TC-0.4  Sink registry"),
    ("TC-1.1 publish",                "TC-1.1  Happy-path publish"),
    ("TC-1.2 sink delivery",          "TC-1.2  Happy-path delivery"),
    ("TC-2.1 condition miss (fired)",  "TC-2.1  Condition miss (p0 fired)"),
    ("TC-2.1 condition miss (no WARN)","TC-2.1  Condition miss (p1 silent)"),
    ("TC-2.2 condition match (count)", "TC-2.2  Condition match (count >= 2)"),
    ("TC-2.2 condition match (WARN)",  "TC-2.2  Condition match (WARN logged)"),
    ("TC-3.1 redaction publish",       "TC-3.1  Redaction publish"),
    ("TC-3.2 userId masked",           "TC-3.2  userId masked to ***"),
    ("TC-3.2 userId raw absent",       "TC-3.2  Raw userId absent"),
    ("TC-3.2 sessionId dropped",       "TC-3.2  sessionId dropped"),
    ("TC-3.2 action preserved",        "TC-3.2  Non-PII field preserved"),
    ("TC-4.1 first publish",           "TC-4.1  First publish"),
    ("TC-4.2 duplicate publish",       "TC-4.2  Duplicate publish (200)"),
    ("TC-4.3 dedup log",               "TC-4.3  Backend dedup log"),
    ("TC-4.4 exactly one delivery",    "TC-4.4  Exactly one delivery"),
]

W = 44
print(f"\n{'═'*67}")
print(f"  AuditFlow Regression Results")
print(f"{'─'*67}")
passed = 0
for key, label in TC_LABELS:
    if key in _results:
        ok, detail = _results[key]
        icon = "✓" if ok else "✗"
        passed += ok
        print(f"  {icon}  {label:<{W}} {detail[:32]}")
    else:
        print(f"  -  {label:<{W}} DID NOT RUN")
print(f"{'─'*67}")
print(f"  TOTAL  {passed}/{len(TC_LABELS)} passed")
print(f"{'═'*67}\n")
